# Week 1 Lab — Introduction & Mathematical Foundations

**MANG2074 Financial Econometrics 1**

**Objectives**

- Load and inspect financial data with `pandas`.
- Transform prices into simple and log returns.
- Compute and interpret summary statistics (mean, volatility, skewness, kurtosis).
- Test for normality with the Jarque–Bera test.

**Data**

- `../data/sp500.csv` — daily S&P 500 index level, 1950–2018 (`date`, `sp500`).
- `../data/ukhp.csv` — monthly UK average house price (Nationwide), 1991–2018 (`Month`, `Average House Price`).


## Task 1 — Load and inspect the data

Read both CSV files. Use the date column as the index and parse it as dates. Then inspect each DataFrame with `.head()`, `.info()` and `.describe()`. How many observations are there? Any missing values?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sp = pd.read_csv('../data/sp500.csv', index_col=0, parse_dates=True)
hp = pd.read_csv('../data/ukhp.csv', index_col=0, parse_dates=True)

# YOUR CODE HERE: inspect both DataFrames (head, info, describe)


## Task 2 — Plot the price levels

Plot the S&P 500 level and the UK average house price over time (two separate figures). Label the axes. What do you notice about the long-run behaviour of both series? Do they look like they fluctuate around a constant mean?

In [ ]:
# YOUR CODE HERE: plot sp['sp500'] and hp['Average House Price']
# Hint: sp['sp500'].plot(figsize=(10,4)); plt.ylabel(...); plt.title(...); plt.show()


## Task 3 — Compute simple and log returns

For the S&P 500, create two new columns:

- `ret_simple` $= 100\,(P_t/P_{t-1} - 1)$ — hint: `pct_change()`
- `ret_log` $= 100\,(\ln P_t - \ln P_{t-1})$ — hint: `np.log(...).diff()`

For the house-price data create the monthly percentage change `dhp` (simple return). Drop the missing first observation in each case.

In [ ]:
sp['ret_simple'] = 100 * sp['sp500'].pct_change()
# YOUR CODE HERE: sp['ret_log'] = ...
# YOUR CODE HERE: hp['dhp'] = ...

sp = sp.dropna()
hp = hp.dropna()
sp.head()


## Task 4 — How different are simple and log returns?

Compute the difference `ret_simple - ret_log` and report its mean, maximum and the dates of the largest discrepancies. When do the two definitions disagree most? (Hint: think about the size of the return on those days.)

In [ ]:
diff = sp['ret_simple'] - sp['ret_log']
# YOUR CODE HERE: describe the difference; find the dates of the largest absolute differences
# Hint: diff.abs().nlargest(5)


## Task 5 — Plot the returns

Plot the daily S&P 500 log returns as a time series. Compare it visually with the price plot from Task 2. Two things to look for: (i) does the series now look mean-reverting (stationary)? (ii) do calm and turbulent periods cluster together (volatility clustering — a preview of Week 7)?

In [ ]:
# YOUR CODE HERE: time-series plot of sp['ret_log']


## Task 6 — Summary statistics

For S&P 500 log returns and for UK house-price growth, compute: mean, standard deviation, skewness and **excess** kurtosis. Then annualise: multiply the daily S&P mean by 252 and its standard deviation by $\sqrt{252}$; multiply the monthly house-price mean by 12 and its standard deviation by $\sqrt{12}$.

In [ ]:
def moments(x, label):
    print(f"--- {label} ---")
    print(f"mean    : {x.mean():8.4f}")
    print(f"std dev : {x.std():8.4f}")
    # YOUR CODE HERE: print skewness (x.skew()) and excess kurtosis (x.kurtosis())

moments(sp['ret_log'], 'S&P500 daily log returns (%)')
moments(hp['dhp'], 'UK house price growth, monthly (%)')

# YOUR CODE HERE: annualised mean and volatility for both series


## Task 7 — Histogram vs the normal distribution

Plot a histogram of the S&P 500 log returns (use `density=True`, around 100 bins) and overlay the normal density with the same mean and standard deviation (`scipy.stats.norm.pdf`). Where does the empirical distribution differ from the normal — centre, shoulders, tails?

In [ ]:
r = sp['ret_log']
x = np.linspace(r.min(), r.max(), 500)

plt.figure(figsize=(10,5))
plt.hist(r, bins=100, density=True, alpha=0.6, label='S&P500 daily log returns')
# YOUR CODE HERE: overlay stats.norm.pdf(x, loc=r.mean(), scale=r.std())
plt.legend(); plt.show()


## Task 8 — Jarque–Bera test

Compute the JB statistic by hand from the formula

$$JB = T\left[\frac{S^2}{6} + \frac{(K-3)^2}{24}\right]$$

using the *raw* (not excess) kurtosis $K$, then verify with `scipy.stats.jarque_bera`. Do this for both series. State $H_0$, the distribution under the null, and your conclusion at the 5% level.

In [ ]:
def jb_by_hand(x):
    T = len(x)
    S = stats.skew(x)
    K = stats.kurtosis(x, fisher=False)   # raw kurtosis (normal = 3)
    # YOUR CODE HERE: return the JB statistic
    pass

for series, label in [(sp['ret_log'], 'S&P500'), (hp['dhp'], 'UK house prices')]:
    jb_stat, jb_p = stats.jarque_bera(series)
    print(f"{label}: JB(scipy) = {jb_stat:.1f}, p-value = {jb_p:.4f}, JB(by hand) = {jb_by_hand(series)}")


## Task 9 — Write up your verdict

In the markdown cell below, answer in 3–5 sentences:

1. Are daily S&P 500 returns normally distributed? Which feature (skewness or kurtosis) drives your conclusion?
2. Why does non-normality matter for a risk manager using a model that assumes normal returns?
3. Why did we analyse returns rather than the index level itself?

*YOUR ANSWER HERE*